# Nyasha LoRA Training
ZIMSEC Math tutor model using Unsloth + LoRA

**How to use:**
1. Runtime → Change runtime type → T4 GPU (free) or A100 (Colab Pro+)
2. Upload `nyasha_alpaca_full.json` to Colab session (from the Files tab)
3. Run all cells
4. For quick test: set USE_TEST_MODE = True first

In [ ]:
# Install dependencies
!pip install -q unsloth[colab] accelerate bitsandbytes xformers trl
!pip install -q datasets transformers peft

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json, os

# ─── CONFIG ──────────────────────────────────────────────
MODEL_NAME = 'unsloth/DeepSeek-R1-Distill-Qwen-7B'
LORA_RANK = 16
MAX_SEQ_LEN = 1024
USE_TEST_MODE = False  # Set True for quick test with 50 samples
# ─────────────────────────────────────────────────────────

In [ ]:
# Upload nyasha_alpaca_full.json via Colab Files tab or run:
from google.colab import files
print('Upload nyasha_alpaca_full.json:')
uploaded = files.upload()

data = json.load(open(list(uploaded.keys())[0]))

if USE_TEST_MODE:
    data = data[:50]
    print(f'Test mode: {len(data)} samples')
else:
    print(f'Full dataset: {len(data)} samples')

In [ ]:
dataset = Dataset.from_list(data)
print(f'Dataset ready: {len(dataset)} examples')
print(f'Sample:\n{data[0]["text"][:200]}...')

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    device_map='auto',
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    max_seq_length=MAX_SEQ_LEN,
)

print('Model loaded with LoRA adapters')

In [ ]:
train_args = TrainingArguments(
    per_device_train_batch_size=2 if USE_TEST_MODE else 4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    save_steps=100,
    output_dir='nyasha-lora-output',
    optim='adamw_8bit',
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    args=train_args,
)

# Train!
trainer.train()
print('✅ Training complete!')

In [ ]:
# Save locally
model.save_pretrained('nyasha-lora')
tokenizer.save_pretrained('nyasha-lora')
print('✅ Model saved to nyasha-lora/')

# Download to your machine
from google.colab import files
import shutil
shutil.make_archive('nyasha-lora', 'zip', 'nyasha-lora')
files.download('nyasha-lora.zip')

In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)
prompt = "You are Nyasha, a ZIMSEC Mathematics tutor.\n\n### Instruction:\nSolve 3x + 7 = 22\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1)
result = tokenizer.decode(outputs[0])
print('=== Output ===')
print(result.split('### Response:')[-1].strip())